In [1]:
import pickle
import os

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from dataset.dataset import load_data
from softmax_CBM.dataset.dataset import load_data as load_data_softmax
from dataset.utils import *
import matplotlib.pyplot as plt

from models.model import *

torch.set_printoptions(sci_mode=False)

import lrp
from models.lrp_converter import convert

from PIL import Image

import threading

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("Device:", device, torch.cuda.get_device_name(0))
else:
    print("Device:", device)

Device: cuda:0 NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [3]:
class Transform():
	"""
	"""

	def __init__(self):
		self.transform = transforms.Compose([
			transforms.Resize((299, 299)),
			transforms.ToTensor(),
			transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[2, 2, 2])
		])

	def __call__(self, img, normalise=True):
		"""
		"""
		if normalise:
			return self.transform(img)
		else:
			return torch.as_tensor(transforms.Resize((299, 299))(img))

class Model():
	def __init__(
		self,
		n_img_out=1,
		use_sig=True,
		n_concepts=52,
		n_classes=6,
		concept_index_to_string="./dataset/playing-cards/splits/three_card_poker_card_class/concepts.txt",
		class_index_to_string="./dataset/playing-cards/splits/three_card_poker_card_class/classes.txt",
		img_transform=Transform(),
		XtoCtoY_path="./saves/XtoCtoY_converted.pth"
		):
		super(Model, self).__init__()

		self.use_sig = use_sig
		self.concept_index_to_string = IndexToString(concept_index_to_string)
		self.class_index_to_string = IndexToString(class_index_to_string)
		self.img_transform = img_transform

		self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
		if torch.cuda.is_available():
			print("Device:", self.device, torch.cuda.get_device_name(0))
		else:
			print("Device:", self.device)


		XtoCtoY = End2EndModel(num_classes=n_classes, num_concepts=n_concepts, num_images_per_output=n_img_out, use_sigmoid=self.use_sig, use_modulelist=False)
		XtoCtoY.load_state_dict(torch.load(XtoCtoY_path, map_location=torch.device('cpu')))
		XtoC = convert(XtoCtoY.x_to_c_model)

		XtoC_model_1 = []
		XtoC_model_2 = []
		XtoC_model_3 = []

		for idx, m in enumerate(XtoC.children()):
			if idx < 15:
				XtoC_model_1.append(m)
			elif idx < 29:
				XtoC_model_2.append(m)
			else:
				XtoC_model_3.append(m)

		self.XtoC_model_1 = lrp.Sequential(*XtoC_model_1)
		self.XtoC_model_2 = lrp.Sequential(*XtoC_model_2)
		self.XtoC_model_3 = lrp.Sequential(*XtoC_model_3)

		self.XtoC_model_1.to(self.device)
		self.XtoC_model_2.to(self.device)
		self.XtoC_model_3.to(self.device)

		self.CtoY = XtoCtoY.c_to_y_model.to(self.device)

		self.raw_input = None
		self.transformed_input = None
		self.input_name = None
		self.input_path = None
		self.pred_concepts = None
		self.task_out = None

		self.thread = threading.Semaphore()

	def predict(self, concepts=None):
		""" pass an input image through a model to get a prediction for task and concepts.
		Optionaly also generate and save concept saliency maps

		args:
			concepts (None / list of floats): List of concepts (only used for C to Y / interveneing)

		return:
			task prediction (string): task class predicted
			concepts (list of floats): list with concept predictions
		"""

		# C to Y only
		if concepts != None:
			c_to_y_input = torch.FloatTensor(concepts).unsqueeze(0).to(self.device)
		# End to end
		else:
			# Transform image and prepare for input to model
			self.transformed_input = self.img_transform(self.raw_input).to(self.device)
			self.transformed_input = self.transformed_input.unsqueeze(0)
			self.transformed_input.requires_grad_(True)
			self.transformed_input.grad = None  # Reset gradient

			# Predict concepts
			output_1 = self.XtoC_model_1.forward(self.transformed_input, explain=True, rule="alpha1beta0")
			output_2 = self.XtoC_model_2.forward(output_1, explain=True, rule="epsilon")
			self.pred_concepts = self.XtoC_model_3.forward(output_2, explain=True, rule="gradient")

			# Prepare concept predictions for task prediction
			outputs = []
			for out in [self.pred_concepts]:
				outputs.append(out)
			if self.use_sig:
				c_to_y_input = torch.nn.Sigmoid()(torch.cat(outputs, dim=1))
				sig_concepts = c_to_y_input
			else:
				c_to_y_input = torch.cat((outputs), dim=1)
				sig_concepts = torch.nn.Sigmoid()(torch.cat(outputs, dim=1))

			concepts = sig_concepts[0].tolist()

		self.task_out = self.CtoY(c_to_y_input)

		_, predicted = self.task_out.max(1)

		print(predicted)

		return self.class_index_to_string(predicted.item()), concepts

	def gen_saliency(self, concept_id, cmap="seismic", **kwargs):
		"""
		Generate and save saleincy map for a given concept

		args:
			concept_id (int): ID of the concept to generate a saleincy map of
			cmap (string): Name of matplotlib colour map to use
			**kwargs: additional variables which are passed directly to save_image()
		"""
		self.thread.acquire()  # add lock to avoid CUDA out of memory error
		filename = f"{concept_id}_{self.input_name}"
		# Only generate saliency map if it does not already exist
		if os.path.exists(f"{os.path.dirname(self.input_path)}/{filename}") == False:
			attr = get_saliency(self.pred_concepts, self.transformed_input, concept_id, cmap_name=cmap)
			save_image(filename=filename, img=attr.squeeze(), type="plt", **kwargs)
		self.thread.release()

		return filename

	def get_readable_concepts(self, concept_vec):
		"""
		Return a list of concept prediction in a readable datastructure

		args:
			concept_vec (list of floats): list of floats. Each item should be indexed to match concept indexes

		return:
			concept_out (list of tuples): list of concept predictions in the form
				(concept ID, concept string, concept value)

		"""
		concept_out = []
		for idx, i in enumerate(concept_vec):
			concept_out.append((idx, self.concept_index_to_string(idx), i))
		return concept_out

	def set_input(self, path):
		"""
		replaces the model input with a new image. Any alpha channel is removed

		args:
			image (string): file path to input image
		"""
		# remove old input + results from model and memory
		del self.raw_input
		del self.transformed_input
		del self.pred_concepts
		del self.task_out

		# add new input to class instance
		self.input_path = path
		self.input_name = os.path.basename(path)
		self.raw_input = Image.open(self.input_path).convert('RGBA')
		background = Image.new('RGBA', self.raw_input.size, (255,255,255))
		self.raw_input = Image.alpha_composite(background, self.raw_input)
		self.raw_input = self.raw_input.convert('RGB')

In [4]:
CURRENT_MODEL = Model()

Device: cuda:0 NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [5]:
CURRENT_MODEL.set_input(os.path.join("dataset", "playing-cards", "imgs", "three_card_poker", "0.png"))
task_out, concepts = CURRENT_MODEL.predict()

tensor([4], device='cuda:0')
{0: 'straight_flush', 1: 'three_of_a_kind', 2: 'straight', 3: 'flush', 4: 'pair', 5: 'high_card'}
